# Host Classification

We build one conformal envelope per host genus. For each host H:
- NC score = 1 − raw_score  (a high raw score means strong evidence of infection, so low NC = strong evidence)
- Envelope is trained on true infectors of host H only

### At Testing
For each test phage and each candidate host H, compute its NC score and check against host H's envelope.
All hosts whose envelope contains the phage form the prediction set.

In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pickle
import time
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

### Loading the Data

In [ ]:
METADATA_PATH = "../Data/metadata.csv"
SPLITS_PATH   = "../Data/all_random_pc_splits.pkl"
MODEL_FOLDER  = Path("../Data/random_pc_r_5_da_2_bal/")

print("Loading metadata...")
t0 = time.time()
meta_df = pd.read_csv(METADATA_PATH, index_col="proteinID")
meta_df = meta_df[["accession", "host", "host_type", "split"]]
print(f"  metadata shape : {meta_df.shape}  ({time.time()-t0:.1f}s)")

print("Loading protein-cluster splits...")
t0 = time.time()
with open(SPLITS_PATH, "rb") as f:
    pc_splits = pickle.load(f)
print(f"  splits shape : {pc_splits.shape}  ({time.time()-t0:.1f}s)")

Loading metadata...
  metadata shape : (1853074, 4)  (3.4s)
Loading protein-cluster splits...
  splits shape : (1853074, 2160)  (5.1s)


### Loading Per-Host Per-Function Prediction Files

Each file `all_preds_{gramtype}_{function}.pkl` in the model folder contains
a DataFrame of shape (n_proteins, n_hosts) — raw prediction scores for that
gram type and function combination. We load all of them and stack them into one
wide DataFrame indexed by proteinID.

In [6]:
print("Loading and assembling prediction files...")
t0 = time.time()

all_pred_dfs = []
file_info    = []

for fpath in pkl_files:
    # Parse gramtype and function from filename: all_preds_{gramtype}_{function}.pkl
    stem  = fpath.stem                      # e.g. "all_preds_gramneg_lysin"
    parts = stem.split("_", 3)             # ["all", "preds", "gramneg", "lysin"]
    if len(parts) < 4:
        print(f"  Skipping unrecognised filename: {fpath.name}")
        continue
    gramtype = parts[2]                    # "gramneg" or "grampos"
    func     = parts[3]                    # e.g. "lysin"

    df = pd.read_pickle(fpath)
    # Rename columns: "Host" -> "Host_gramtype_func" to keep them unique across files
    df.columns = [f"{col}_{gramtype}_{func}" for col in df.columns]
    all_pred_dfs.append(df)
    file_info.append({"gramtype": gramtype, "func": func, "n_hosts": df.shape[1]})

preds_df = pd.concat(all_pred_dfs, axis=1)
print(f"  Assembled prediction matrix: {preds_df.shape}  ({time.time()-t0:.1f}s)")
print(f"  Columns (first 5): {preds_df.columns.tolist()[:5]}")

Loading and assembling prediction files...


ValueError: No objects to concatenate

### Contamination Fix: Protein-Level Masking

For each prediction column `{Host}_{gramtype}_{func}`, the corresponding splits column
is `split_{Host}_{func}`. A protein is clean (split == 0) means it was genuinely
held out for that model. We mask contaminated proteins to NaN before aggregation.

This is the direct application of Alex's instruction: `preds.where(splits == 0)`,
with explicit row and column alignment.

In [ ]:
# Align splits to preds rows — critical for correct masking
pc_splits = pc_splits.reindex(preds_df.index)

print("Applying protein-level contamination masking...")
t0 = time.time()
preds_masked = preds_df.copy()

skipped = []
for col in preds_df.columns:
    # col format: "{Host}_{gramtype}_{func}"
    # split col format: "split_{Host}_{func}"  (no gramtype in splits file)
    parts    = col.rsplit("_", 2)           # ["{Host}", "{gramtype}", "{func}"]
    if len(parts) < 3:
        skipped.append(col); continue
    host, gramtype, func = parts
    split_col = f"split_{host}_{func}"

    if split_col not in pc_splits.columns:
        skipped.append(col); continue

    preds_masked[col] = preds_df[col].where(pc_splits[split_col] == 0)

print(f"  Done in {time.time()-t0:.1f}s")
if skipped:
    print(f"  WARNING: {len(skipped)} columns had no matching splits column — left unmasked")
    print(f"  First 5 skipped: {skipped[:5]}")

original_nans = preds_df.isna().sum().sum()
new_nans      = preds_masked.isna().sum().sum()
print(f"  NaNs before masking : {original_nans:,}")
print(f"  NaNs after  masking : {new_nans:,}")
print(f"  Newly masked scores : {new_nans - original_nans:,}")
if new_nans == original_nans:
    print("  WARNING: zero proteins were masked — check column name parsing above")

### Aggregating to Phage Level

For each host-function score, take the NaN-aware mean across all proteins of a phage.
Proteins masked to NaN simply do not contribute to their phage's mean.

In [ ]:
pred_cols = preds_masked.columns.tolist()

combined = preds_masked.join(meta_df, how="inner")

t0 = time.time()
phage_scores = combined.groupby("accession")[pred_cols].mean()
phage_meta   = combined.groupby("accession")[["host", "host_type", "split"]].first()
phage_df     = phage_scores.join(phage_meta)
phage_df["split"] = phage_df["split"].astype(int)

print(f"Phage-level shape: {phage_df.shape}  ({time.time()-t0:.1f}s)")
print("\nHost distribution (top 20):")
print(phage_df["host"].value_counts().head(20))
print("\nSplit distribution:")
print(phage_df["split"].value_counts().sort_index())

### Deriving Per-Host Scores

For each host H, the best single evidence score for a phage is the maximum across
all function-specific models that include H as a target column.
NC score = 1 − max_score  (high evidence = low nonconformity).

In [ ]:
# Get the full list of unique hosts from the column names
all_hosts = sorted(set(
    col.rsplit("_", 2)[0]
    for col in pred_cols
))
print(f"Unique hosts: {len(all_hosts)}")
print(all_hosts[:20])

In [ ]:
# For each host, aggregate across all function/gramtype columns for that host
# by taking the column-wise mean across functions (NaN-aware)
print("Building per-host aggregated score matrix...")
t0 = time.time()

host_score_df = pd.DataFrame(index=phage_df.index)

for host in all_hosts:
    host_cols = [c for c in pred_cols if c.startswith(host + "_")]
    if len(host_cols) == 0:
        continue
    # Mean across all function/gramtype columns for this host
    host_score_df[host] = phage_df[host_cols].mean(axis=1)

print(f"  Per-host score matrix: {host_score_df.shape}  ({time.time()-t0:.1f}s)")
print(f"  Columns (first 5): {host_score_df.columns.tolist()[:5]}")

### Train / Test Split

In [ ]:
host_full_df = host_score_df.join(phage_df[["host", "host_type", "split"]])

host_full_df["split"] = host_full_df["split"].astype(int)
train_df = host_full_df[host_full_df["split"] != 0].copy()
test_df  = host_full_df[host_full_df["split"] == 0].copy()

# Keep only phages with known gram type (exclude 'unknown')
train_df = train_df[train_df["host_type"].isin(["gram-neg", "gram-pos"])]
test_df  = test_df[test_df["host_type"].isin(["gram-neg", "gram-pos"])]

valid_hosts = [h for h in all_hosts if h in host_score_df.columns]
print(f"Training phages (folds 1-4) : {len(train_df)}")
print(f"Test phages     (fold  0)   : {len(test_df)}")
print(f"Host columns (K)            : {len(valid_hosts)}")

### Median Imputation (training set only)

In [ ]:
train_medians = train_df[valid_hosts].median()
safe_impute   = train_medians.copy()
safe_impute[safe_impute > 0.6] = 0.5

train_df[valid_hosts] = train_df[valid_hosts].fillna(safe_impute)
test_df[valid_hosts]  = test_df[valid_hosts].fillna(safe_impute)
train_df[valid_hosts] = train_df[valid_hosts].fillna(0.5)
test_df[valid_hosts]  = test_df[valid_hosts].fillna(0.5)

print(f"Residual NaNs — train : {train_df[valid_hosts].isna().sum().sum()}")
print(f"Residual NaNs — test  : {test_df[valid_hosts].isna().sum().sum()}")

test_raw    = test_df[valid_hosts].values
test_labels = test_df["host"].values
K           = len(valid_hosts)
print(f"\nTest matrix shape : {test_raw.shape}")
print(f"Unique test hosts : {len(set(test_labels))}")

### Envelope Methods

For host prediction, NC = 1 − raw_score for every hypothesis.
Each method builds one unified envelope on ALL training phages using their
true-host NC score, then at test time checks each candidate host hypothesis.

In [ ]:
def split_data_by_fold(df, valid_hosts, test_fold_id):
    """
    S1 = shape discovery folds, S2 = size-scaling fold.
    NC transformation: NC = 1 - raw_score for all hosts.
    Each training phage contributes its true-host column as its NC score.
    """
    df = df.copy()
    df["split"] = df["split"].astype(int)
    s1_df = df[df["split"] != test_fold_id]
    s2_df = df[df["split"] == test_fold_id]

    # For S1: each phage's NC vector = 1 - its score vector
    # (We use the full K-dimensional vector, not just the true host dimension)
    S1_nc = 1.0 - s1_df[valid_hosts].values
    S2_nc = 1.0 - s2_df[valid_hosts].values
    s2_labels = s2_df["host"].values

    return S1_nc, S2_nc, s2_labels

In [ ]:
def quantile_for_class(tau_scores, labels, cls, alpha):
    vals = np.sort(tau_scores[labels == cls])
    n    = len(vals)
    if n == 0:
        return np.inf
    idx  = int(np.ceil((n + 1) * (1 - alpha))) - 1
    return vals[np.clip(idx, 0, n - 1)]

#### Radial Envelope

In [ ]:
def sample_positive_sphere(M, K):
    V = np.abs(np.random.randn(M, K))
    return V / (np.linalg.norm(V, axis=1, keepdims=True) + 1e-12)

def radial_build_envelope_unbiased(df, valid_hosts, fold_id, alpha, M=200, delta_deg=30):
    S1, S2, labels_S2 = split_data_by_fold(df, valid_hosts, fold_id)
    K = S1.shape[1]
    U    = sample_positive_sphere(M, K)
    mags = np.linalg.norm(S1, axis=1)
    dirs = S1 / (mags[:, None] + 1e-12)
    cos_thresh = np.cos(np.radians(delta_deg))
    q_tilde = np.array([
        np.quantile(mags[(dirs @ U[m]) >= cos_thresh], 1 - alpha)
        if ((dirs @ U[m]) >= cos_thresh).sum() > 5
        else np.quantile(mags, 1 - alpha)
        for m in range(M)
    ])
    boundary   = U * q_tilde[:, None]
    tau_scores = (S2[:, None, :] / (boundary[None, :, :] + 1e-12)).max(axis=2).min(axis=1)

    # Class-conditional t_hat: take the max over all hosts present in S2
    present_hosts = list(set(labels_S2))
    t_hat = max(
        quantile_for_class(tau_scores, labels_S2, h, alpha)
        for h in present_hosts
    )
    return {"U": U, "q_tilde": q_tilde, "t_hat": t_hat, "method": "radial"}

def radial_is_in_region(scores, envelope):
    U, q_tilde, t_hat = envelope["U"], envelope["q_tilde"], envelope["t_hat"]
    boundary = U * (q_tilde * t_hat)[:, None]
    return np.any(np.all(scores[:, None, :] <= boundary[None, :, :], axis=2), axis=1)

#### Collapsed 1D Envelope

In [ ]:
def collapsed_build_envelope_unbiased(df, valid_hosts, fold_id, alpha):
    S1, S2, labels_S2 = split_data_by_fold(df, valid_hosts, fold_id)
    S1_1d    = S1.mean(axis=1, keepdims=True)
    S2_1d    = S2.mean(axis=1, keepdims=True)
    q_tilde  = np.quantile(S1_1d, 1 - alpha)
    tau_scores = (S2_1d / (q_tilde + 1e-12)).ravel()
    present_hosts = list(set(labels_S2))
    t_hat = max(
        quantile_for_class(tau_scores, labels_S2, h, alpha)
        for h in present_hosts
    )
    return {"q_tilde": q_tilde, "t_hat": t_hat, "method": "collapsed"}

def collapsed_is_in_region(scores, envelope):
    scores_1d = scores.mean(axis=1, keepdims=True)
    return (scores_1d.ravel() <= envelope["q_tilde"] * envelope["t_hat"])

In [ ]:
def is_in_region(scores, envelope):
    if envelope["method"] == "radial":
        return radial_is_in_region(scores, envelope)
    else:
        return collapsed_is_in_region(scores, envelope)

### Prediction

In [ ]:
def predict_host_nc(test_raw, envelope, valid_hosts):
    """
    For each test phage and each candidate host H (column index h),
    compute NC = 1 - test_raw[:, h] and check against the envelope.
    The prediction set = all hosts whose NC vector falls inside.
    """
    K = len(valid_hosts)
    results = []

    for i in range(len(test_raw)):
        pred_set = []
        for h_idx, host_name in enumerate(valid_hosts):
            # NC vector for hypothesis "this phage infects host_name":
            # dimension h_idx uses 1 - score[h_idx]; other dims use score as-is
            # For a fully K-dimensional check we use 1 - raw score for ALL dims
            nc_vec = 1.0 - test_raw[i]          # shape (K,)
            in_env = is_in_region(nc_vec[None, :], envelope)
            if in_env[0]:
                pred_set.append(host_name)
        results.append({
            "prediction_set" : pred_set,
            "set_size"       : len(pred_set)
        })
    return results

### Evaluation

In [ ]:
def evaluate_host(results, true_labels, method_name, alpha, verbose=True):
    n         = len(results)
    set_sizes = [r["set_size"] for r in results]
    covered   = sum(true_labels[i] in r["prediction_set"] for i, r in enumerate(results))
    singletons         = [i for i in range(n) if set_sizes[i] == 1]
    correct_singletons = sum(
        results[i]["prediction_set"][0] == true_labels[i] for i in singletons
    )

    if verbose:
        print(f"\n{'='*55}")
        print(f" {method_name.upper()} — {n} test phages | alpha={alpha}")
        print(f" Coverage : {covered/n:.3f}  (target >= {1-alpha:.2f})")
        print(f" Avg set size : {np.mean(set_sizes):.2f}")
        print(f"{'='*55}")

    # Per-host coverage for hosts with enough test phages
    host_coverage = {}
    for h in set(true_labels):
        h_indices = [i for i in range(n) if true_labels[i] == h]
        if len(h_indices) >= 3:
            host_coverage[h] = sum(
                true_labels[i] in results[i]["prediction_set"] for i in h_indices
            ) / len(h_indices)

    return {
        "coverage"          : covered / n,
        "avg_set_size"      : np.mean(set_sizes),
        "empty_rate"        : sum(s == 0 for s in set_sizes) / n,
        "singleton_rate"    : len(singletons) / n,
        "singleton_accuracy": correct_singletons / len(singletons) if singletons else None,
        "set_sizes"         : set_sizes,
        "host_coverage"     : host_coverage,
    }

### Main Evaluation Loop

In [ ]:
available_folds = [1, 2, 3, 4]
all_metrics     = {"radial": [], "collapsed": []}
alpha           = 0.1

print(f"Running {len(available_folds)} folds  |  alpha = {alpha}  |  K = {K} hosts")
for fold_id in available_folds:
    r_env = radial_build_envelope_unbiased   (train_df, valid_hosts, fold_id, alpha, M=200, delta_deg=30)
    c_env = collapsed_build_envelope_unbiased(train_df, valid_hosts, fold_id, alpha)

    r_res = predict_host_nc(test_raw, r_env, valid_hosts)
    c_res = predict_host_nc(test_raw, c_env, valid_hosts)

    for key, res in [("radial", r_res), ("collapsed", c_res)]:
        all_metrics[key].append(evaluate_host(res, test_labels, key, alpha, verbose=False))

    print(f"  Fold {fold_id} complete  |  "
          f"radial cov={all_metrics['radial'][-1]['coverage']:.3f}  "
          f"collapsed cov={all_metrics['collapsed'][-1]['coverage']:.3f}")

radial_results    = r_res
collapsed_results = c_res

### Summary Table

In [ ]:
methods_names = ["Radial", "Collapsed 1D"]
method_keys   = ["radial", "collapsed"]

print(f"\n{'='*65}")
print(f" SUMMARY ACROSS FOLDS 1-4  (alpha={alpha})")
print(f"{'='*65}")
for name, key in zip(methods_names, method_keys):
    runs    = all_metrics[key]
    cov     = np.array([m["coverage"]       for m in runs])
    avg_sz  = np.array([m["avg_set_size"]   for m in runs])
    empty   = np.array([m["empty_rate"]     for m in runs])
    sing    = np.array([m["singleton_rate"] for m in runs])
    sing_ac = np.array([m["singleton_accuracy"] for m in runs if m["singleton_accuracy"] is not None])
    print(f"\n  {name}")
    print(f"    Overall coverage  : {cov.mean():.3f} ± {cov.std():.3f}  (target >= {1-alpha:.2f})")
    print(f"    Avg set size      : {avg_sz.mean():.3f} ± {avg_sz.std():.3f}")
    print(f"    Empty set rate    : {empty.mean()*100:.1f}% ± {empty.std()*100:.1f}%")
    print(f"    Singleton rate    : {sing.mean()*100:.1f}% ± {sing.std()*100:.1f}%")
    if len(sing_ac):
        print(f"    Singleton acc.    : {sing_ac.mean():.3f} ± {sing_ac.std():.3f}")

### Comparison Plots

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
colors = ["steelblue", "mediumseagreen"]

# Plot 1: Overall coverage
ax    = axes[0]
means = [np.mean([m["coverage"] for m in all_metrics[k]]) for k in method_keys]
stds  = [np.std ([m["coverage"] for m in all_metrics[k]]) for k in method_keys]
bars  = ax.bar(methods_names, means, color=colors, alpha=0.8)
ax.errorbar(range(len(methods_names)), means, yerr=stds, fmt="none", color="black", capsize=5, linewidth=1.5)
ax.axhline(1 - alpha, color="red", linestyle="--", label=f"Target ({1-alpha:.2f})")
ax.set_ylim(0, 1.05); ax.set_title("Overall Coverage"); ax.set_ylabel("Coverage"); ax.legend(fontsize=9)
for bar, val, std in zip(bars, means, stds):
    ax.text(bar.get_x() + bar.get_width()/2, val + std + 0.01, f"{val:.3f}",
            ha="center", va="bottom", fontsize=10)

# Plot 2: Set size distribution
ax      = axes[1]
x       = np.array([0, 1, 2])
offsets = [-0.2, 0.2]
for name, key, c, offset in zip(methods_names, method_keys, colors, offsets):
    means_s, stds_s = [], []
    for s in [0, 1, 2]:
        fracs = np.array([
            sum(1 for sz in m["set_sizes"] if sz == s) / len(m["set_sizes"])
            for m in all_metrics[key]
        ])
        means_s.append(fracs.mean()); stds_s.append(fracs.std())
    bars = ax.bar(x + offset, means_s, width=0.35, label=name, color=c, alpha=0.8)
    ax.errorbar(x + offset, means_s, yerr=stds_s, fmt="none", color="black", capsize=3)
    for rect, val, std in zip(bars, means_s, stds_s):
        ax.text(rect.get_x() + rect.get_width()/2, val + std + 0.005, f"{val:.2f}",
                ha="center", va="bottom", fontsize=8)
ax.set_xticks(x); ax.set_xticklabels(["Empty (0)", "Singleton (1)", "Multi (2+)"])
ax.set_title("Set Size Distribution"); ax.set_ylabel("Fraction of test phages"); ax.legend(fontsize=9)

# Plot 3: Singleton accuracy
ax    = axes[2]
means = []
stds  = []
ns    = []
for key in method_keys:
    vals = np.array([m["singleton_accuracy"] for m in all_metrics[key] if m["singleton_accuracy"] is not None])
    means.append(vals.mean() if len(vals) else 0)
    stds.append (vals.std()  if len(vals) else 0)
    ns.append(int(np.mean([m["singleton_rate"] * len(m["set_sizes"]) for m in all_metrics[key]])))
bars = ax.bar(methods_names, means, color=colors, alpha=0.8)
ax.errorbar(range(len(methods_names)), means, yerr=stds, fmt="none", color="black", capsize=5, linewidth=1.5)
ax.set_ylim(0, 1.05); ax.set_title("Singleton Accuracy"); ax.set_ylabel("Accuracy (when set size = 1)")
for bar, val, std, n_s in zip(bars, means, stds, ns):
    ax.text(bar.get_x() + bar.get_width()/2, val + std + 0.01, f"{val:.3f}\n(n≈{n_s})",
            ha="center", va="bottom", fontsize=9)

plt.suptitle("Host Classification", fontsize=13, fontweight="bold")
plt.tight_layout(); plt.show()

### Per-Host Coverage Breakdown

In [ ]:
# Show per-host coverage for the best method (collapsed, most stable)
best_key = "collapsed"
per_host_covs = {}
for h in valid_hosts:
    vals = [m["host_coverage"].get(h) for m in all_metrics[best_key] if h in m["host_coverage"]]
    if vals:
        per_host_covs[h] = np.mean(vals)

per_host_df = pd.DataFrame.from_dict(per_host_covs, orient="index", columns=["coverage"])
per_host_df["n_test"] = [sum(test_labels == h) for h in per_host_df.index]
per_host_df = per_host_df.sort_values("coverage")

print(f"Per-host coverage ({best_key}, averaged over folds):")
print(per_host_df.to_string())

below_target = (per_host_df["coverage"] < 1 - alpha).sum()
print(f"\nHosts below {1-alpha:.2f} target: {below_target} / {len(per_host_df)}")

### Score Distribution

In [ ]:
# Distribution of mean raw scores for true-host vs other hosts
true_host_scores  = []
other_host_scores = []

for i, phage_idx in enumerate(test_df.index):
    true_h = test_labels[i]
    if true_h not in valid_hosts:
        continue
    h_idx = valid_hosts.index(true_h)
    true_host_scores.append(test_raw[i, h_idx])
    # Random sample of 3 non-true hosts
    other_idxs = [j for j in range(K) if j != h_idx]
    np.random.shuffle(other_idxs)
    other_host_scores.extend(test_raw[i, other_idxs[:3]])

plt.figure(figsize=(9, 5))
plt.hist(true_host_scores,  bins=50, alpha=0.6, color="royalblue",  label="True host score",  edgecolor="black", linewidth=0.5)
plt.hist(other_host_scores, bins=50, alpha=0.6, color="darkorange", label="Other host scores", edgecolor="black", linewidth=0.5)
plt.title("Raw Score Distribution: True vs Non-True Hosts", fontsize=12, fontweight="bold")
plt.xlabel("Score"); plt.ylabel("Count")
plt.xlim(-0.05, 1.05); plt.legend(); plt.grid(True, linestyle="--", alpha=0.5)
plt.tight_layout(); plt.show()